# First-order local uncertainty for a surface mechanism

## With and without correlations through BEEF-vdW DFT ensemble


**Uncorrelated Analysis:**

The uncorrelated first-order local variance $(\Delta \ln c_i)^2$, for the concentration of species $i$ is:

$$(\Delta \ln c_i)^2 = \sum_{\mathrm{reactions}\; m} \left(\frac{\partial\ln c_i}{\partial\ln k_m}\right)^2 (\Delta \ln k_m)^2  + \sum_{\mathrm{species}\; n} \left(\frac{\partial\ln c_i}{\partial G_n}\right)^2(\Delta G_n)^2$$

We can get the derivatives $\frac{\partial\ln c_i}{\partial\ln k_m}$ and $\frac{\partial\ln c_i}{\partial G_n}$ through sensitivity analysis.

We compute the input uncertainties $\Delta G_n$ and $\Delta \ln k_m$ by looking up the parameter source and adding the uncertainty that corresponds to that method. See the local_uncertainty.ipynb example or [Gao CW, Liu M, Green WH. Uncertainty analysis of correlated parameters in automated reaction mechanism generation. International journal of chemical kinetics. 2020;52(4):266-282. doi:10.1002/kin.21348](https://onlinelibrary.wiley.com/doi/10.1002/kin.21348) for more information.


**Correlated Analysis:**

The correlated first-order local variance $(\Delta \ln c_i)^2$, for the concentration of species $i$ is:

$$(\Delta \ln x)^2 = \sum_w \sum_{w'} \left(\sum_m \frac{\partial \ln x}{\partial \ln k_m} \frac{\partial \ln k_m}{\partial q_w} + \sum_n \frac{\partial \ln x}{\partial G_n} \frac{\partial G_n}{\partial q_w}\right) \Sigma_{ww'} \left(\sum_{m'} \frac{\partial \ln x}{\partial \ln k_{m'}} \frac{\partial \ln k_{m'}}{\partial q_{w'}} + \sum_{n'} \frac{\partial \ln x}{\partial G_{n'}} \frac{\partial G_{n'}}{\partial q_{w'}}\right)$$

Here, we utilize underlying input parameters $q_w$ which may be correlated to each other. For example, species thermo libraries computed using BEEF-vdW have known correlations through the DFT ensemble.

These covariances are computed and saved in the covariance matrix $\Sigma_{ww'}$.

The derivatives $\frac{\partial\ln c_i}{\partial\ln k_m}$ and $\frac{\partial\ln c_i}{\partial G_n}$ are computed through sensitivity analysis, same as the uncorrelated analysis. 

Finally, the derivatives $\frac{\partial\ln k_m}{\partial q_w}$ and $\frac{\partial\ln G_n}{\partial q_w}$ describe the weight of each underlying parameter's contribution to a particular species Gibbs energy or reaction rate, and these are tallied up by the uncertainty tool.

In [ ]:
import os

from IPython.display import display, Image

import rmgpy
from rmgpy.tools.uncertainty import Uncertainty, process_local_results
from rmgpy.tools.canteramodel import get_rmg_species_from_user_species
from rmgpy.species import Species

import logging
logging.basicConfig()
logging.getLogger().setLevel(logging.WARNING)

import matplotlib.pyplot as plt
%matplotlib inline

## 1. Load the mechanism (CH4 + O2 over Pt111) and reactor settings

In [ ]:
# This is a mechanism of methane partial oxidation over Pt111

# Must use annotated chemkin file
chemkin_file = 'data/cpox_methane/chem_annotated-gas.inp'
surface_chemkin_file = 'data/cpox_methane/chem_annotated-surface.inp'
dict_file = 'data/cpox_methane/species_dictionary.txt'

# Initialize the Uncertainty class instance and load the model
uncertainty = Uncertainty(output_directory='./temp/surface_uncertainty')
uncertainty.load_model(chemkin_file, dict_file, surface_path=surface_chemkin_file)

# Map the species to the objects within the Uncertainty class
methane = Species().from_smiles('C')
X = Species().from_smiles('*')  # surface site
Ar = Species().from_smiles('[Ar]')
O2 = Species().from_smiles('[O][O]')
H2O = Species().from_smiles('O')
CO2 = Species().from_smiles('O=C=O')
mapping = get_rmg_species_from_user_species([methane, O2, H2O, CO2, Ar, X], uncertainty.species_list)

# Define the reaction conditions
initial_mole_fractions = {mapping[methane]: 0.1, mapping[Ar]: 0.8, mapping[O2]: 0.1}
initial_surface_coverage = {mapping[X]: 1.0}
T = (800, 'K')
P = (1, 'atm')
termination_time = (1.0, 's')
surface_volume_ratio = (1e5, 'm^-1')
surface_site_density = (2.4830E-09, 'mol/(cm^2)')

# Pick the species to do sensitivity analysis for (which output concentrations to analyze)
sensitive_species=[mapping[methane], mapping[CO2]]

INFO:root:(The line it would be on is '    300.000  1000.000  5000.000\n' but that is not formatted as such)


INFO:root:(It should have Tmin in columns 1-10, Tmid in columns 11-20, and Tmax in columns 21-30)


INFO:root:Ignoring short but non-empty line: '    300.000  1000.000  5000.000\n'


## 2. Run Sensitivity Analysis
This computes the derivatives $\frac{\partial \ln c_i}{\partial G_n}$ and $\frac{\partial \ln c_i}{\partial \ln k_m}$ for every input species Gibbs energy $G_n$ and reaction rate $\ln k_m$ in the mechanism with respect to each sensitive species concentration $c_i$

These sensitivities will be used later to propagate the uncertain inputs through the model.

In [ ]:
# Perform the sensitivity analysis- this takes ~30 seconds
uncertainty.sensitivity_analysis(
    initial_mole_fractions,
    sensitive_species,
    T,
    P,
    termination_time,
    number=5,
    fileformat='.png',
    initial_surface_coverages=initial_surface_coverage,
    surface_volume_ratio=surface_volume_ratio,
    surface_site_density=surface_site_density,
)

In [ ]:
# Show the sensitivity plots
for species in sensitive_species:
    print('{}: Reaction Sensitivities'.format(species))
    index = species.index
    display(Image(filename=os.path.join(uncertainty.output_directory,'solver','sensitivity_1_SPC_{}_reactions.png'.format(index))))
    
    print('{}: Thermo Sensitivities'.format(species))
    display(Image(filename=os.path.join(uncertainty.output_directory,'solver','sensitivity_1_SPC_{}_thermo.png'.format(index))))

## 3. Assign (Uncorrelated) Parameter Uncertainties

Here we look up the sources used to compute each parameter and assign a value for $\Delta G_n$ or $\Delta \ln k_m$

### Load the same database that's used in the RMG input file

In [ ]:
uncertainty.load_database(
    thermo_libraries=['surfaceThermoPt111', 'primaryThermoLibrary', 'thermo_DFT_CCSDTF12_BAC', 'DFT_QCI_thermo'],
    kinetics_families=['default', 'surface'],
    reaction_libraries=['Surface/CPOX_Pt/Deutschmann2006_adjusted', 'BurkeH2O2inArHe'],
)

INFO:root:Loading transport library from NOx2018.py in /home/moon/rmg/RMG-database/input/transport/libraries...


INFO:root:Loading transport library from GRI-Mech.py in /home/moon/rmg/RMG-database/input/transport/libraries...


INFO:root:Loading transport library from NIST_Fluorine.py in /home/moon/rmg/RMG-database/input/transport/libraries...


INFO:root:Loading transport group database from /home/moon/rmg/RMG-database/input/transport/groups...


INFO:root:Loading kinetics library Surface/CPOX_Pt/Deutschmann2006_adjusted from /home/moon/rmg/RMG-database/input/kinetics/libraries/Surface/CPOX_Pt/Deutschmann2006_adjusted/reactions.py...


...

INFO:root:Loading thermodynamics library from DFT_QCI_thermo.py in /home/moon/rmg/RMG-database/input/thermo/libraries...


INFO:root:Loading thermodynamics group database from /home/moon/rmg/RMG-database/input/thermo/groups...


INFO:root:Loading thermodynamics SIDTs from /home/moon/rmg/RMG-database/input/thermo/sidt...


ThermoData(Tdata=([300,400,500,600,800,1000,1500],'K'), Cpdata=([60.2599,68.0494,74.7775,80.9311,90.8846,97.4337,105.393],'J/(mol*K)'), H298=(-477.191,'kJ/mol'), S298=(269.551,'J/(mol*K)'), Cp0=(33.2579,'J/(mol*K)'), CpInf=(103.931,'J/(mol*K)'), comment="""Thermo group additivity estimation: group(O2s-(Cds-Cd)(Cds-Cd)) + group(O2s-(Cds-O2d)H) + group(Cds-OdOsOs) + group(Li-OCOdO) + radical(OC=OOJ)""").
The thermo for this species is probably wrong! Setting CpInf = Cphigh for Entropy calculationat T = 2000.0 K...


ThermoData(Tdata=([300,400,500,600,800,1000,1500],'K'), Cpdata=([60.2599,68.0494,74.7775,80.9311,90.8846,97.4337,105.393],'J/(mol*K)'), H298=(-477.191,'kJ/mol'), S298=(269.551,'J/(mol*K)'), Cp0=(33.2579,'J/(mol*K)'), CpInf=(103.931,'J/(mol*K)'), comment="""Thermo group additivity estimation: group(O2s-(Cds-Cd)(Cds-Cd)) + group(O2s-(Cds-O2d)H) + group(Cds-OdOsOs) + group(Li-OCOdO) + radical(OC=OOJ)""").
The thermo for this species is probably wrong! Setting CpInf = Cphigh for Entropy calculationat T = 1666.6666666666665 K...


### Assign input parameter uncertainties according to their sources

In [ ]:
# reset the correlated libraries/groups back to the uncorrelated defaults in case someone runs the cells out of order
uncertainty.thermo_covariance_libraries = None  # default is None
uncertainty.thermo_covariance_groups = None  # these are the defaults

uncertainty.extract_sources_from_model()
uncertainty.assign_intermediate_uncertainties(correlated=False)

# Save the covariance matrix for later comparison
uncorrelated_thermo_cov = uncertainty.get_thermo_covariance_matrix()
uncorrelated_kinetics_cov = uncertainty.get_kinetic_covariance_matrix()


# 4. Propagate the input uncertainties through the mechanism local analysis and show the resulting sensitivity index

Use the first-order local uncertainty equation to get the overall variance in concentration:
$$(\Delta \ln c_i)^2 = \sum_{\mathrm{reactions}\; m} \left(\frac{\partial\ln c_i}{\partial\ln k_m}\right)^2 (\Delta \ln k_m)^2  + \sum_{\mathrm{species}\; n} \left(\frac{\partial\ln c_i}{\partial G_n}\right)^2(\Delta G_n)^2$$

The Sensitivity Index shows the % an input parameter contributes to the overall variance of the output

In [ ]:
result = uncertainty.local_analysis_intermediate(sensitive_species, correlated=False, number=10, fileformat='.png')
print(process_local_results(result, sensitive_species, number=5)[1])

### Repeat for Correlated Uncertainties

The next cell computes correlated uncertainties of the underlying parameters and saves them in $\Sigma_{ww'}$

It also computes the derivatives $\frac{\partial \ln k_m}{\partial q_w}$ and $\frac{\partial \ln G_n}{\partial q_w}$

We use the same sensitivities computed in Step 2.

Then the correlated first-order local variance $(\Delta \ln c_i)^2$, for the concentration of species $i$ is:

$$(\Delta \ln c_i)^2 = \sum_w \sum_{w'} \left(\sum_m \frac{\partial \ln c_i}{\partial \ln k_m} \frac{\partial \ln k_m}{\partial q_w} + \sum_n \frac{\partial \ln c_i}{\partial G_n} \frac{\partial G_n}{\partial q_w}\right) \Sigma_{ww'} \left(\sum_{m'} \frac{\partial \ln c_i}{\partial \ln k_{m'}} \frac{\partial \ln k_{m'}}{\partial q_{w'}} + \sum_{n'} \frac{\partial \ln c_i}{\partial G_{n'}} \frac{\partial G_{n'}}{\partial q_{w'}}\right)$$

In [ ]:
uncertainty.thermo_covariance_libraries = [os.path.join(rmgpy.settings['database.directory'], 'thermo', 'uncertainty', 'surfaceThermoPt111')]
uncertainty.thermo_covariance_groups = [os.path.join(rmgpy.settings['database.directory'], 'thermo', 'uncertainty', 'adsorptionPt111')]

uncertainty.extract_sources_from_model()
uncertainty.assign_intermediate_uncertainties(correlated=True)

# Save the covariance matrix for later comparison
correlated_thermo_cov = uncertainty.get_thermo_covariance_matrix()
correlated_kinetics_cov = uncertainty.get_kinetic_covariance_matrix()


In [ ]:
result = uncertainty.local_analysis_intermediate(sensitive_species, correlated=True, number=10, fileformat='.png')
print(process_local_results(result, sensitive_species, number=5)[1])

Total variance [(d ln(c))^2] for species CH4 is 3.248625
--------------------------------------------------------------------------------
Top  5 reaction rate contributors                              Sensitivity Index
--------------------------------------------------------------------------------
Surface Rate Rule Surface_Abstraction_vdW O-R;*=C=R                      2.4434%
Surface_Library COX(23)<=>X(1)+CO(7)                                     1.0319%
Estimation Nonexact H2OX(31)+CHX(27)<=>HX(21)+XCHOH(81)                  0.2719%
Estimation Family H2OX(31)+CHX(27)<=>HX(21)+XCHOH(81)                    0.2450%
Surface_Library X(1)+H2OX(31)<=>HX(21)+OHX(30)                           0.1869%
--------------------------------------------------------------------------------
Top  5 thermochemistry contributors                            Sensitivity Index
--------------------------------------------------------------------------------
Surface_Library X(1)                                

# 5. We can also export covariance matrices of the input parameter uncertainties

Here we compute the species covariance matrix $\Sigma_G$ and the reaction covariance matrix $\Sigma_{\ln k}$ - note this is different from the covariance matrix for the underlying parameters $\Sigma_{ww'}$, which represents sources that might be reused across species/reactions like individual Benson groups or rate rules.

The species Gibbs covariance matrix $\Sigma_G$ and reaction rate covariance matrix $\Sigma_{\ln k}$ can be useful for uncertainty quantification performed outside of RMG. They can also be useful for optimization and parameter fitting, which can used correlated uncertainties to find the best fit to experimental data.


For the uncorrelated covariance matrices on the left, only the diagonal values are non-zero.
The correlated analysis for the thermo on the right includes correlations through the BEEF-vdW ensemble and the reference species that are used to compute each species.

The gas-phase species are listed first in the mechanism, and these have much lower uncertainty values than the surface-phase species because the DFT is more accurate for gas-phase species than for adsorbates

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 6))
plt.subplots_adjust(wspace=0.3)

# Uncorrelated Species Thermo Uncertainties
mat1 = axs[0].matshow(uncorrelated_thermo_cov)
axs[0].xaxis.set_ticks_position('bottom')
axs[0].xaxis.set_label_position('bottom')
cb1 = fig.colorbar(mat1, ax=axs[0], fraction=0.046, pad=0.04)
cb1.ax.set_title('    (kcal/mol)^2')
axs[0].set_title('Uncorrelated\nThermo Covariance Matrix')
axs[0].set_xlabel('Species Index')
axs[0].set_ylabel('Species Index')

# Correlated Species Thermo Uncertainties
mat2 = axs[1].matshow(correlated_thermo_cov)
axs[1].xaxis.set_ticks_position('bottom')
axs[1].xaxis.set_label_position('bottom')
cb2 = fig.colorbar(mat2, ax=axs[1], fraction=0.046, pad=0.04)
cb2.ax.set_title('    (kcal/mol)^2')
axs[1].set_title('Correlated\nThermo Covariance Matrix')
axs[1].set_xlabel('Species Index')
axs[1].set_ylabel('Species Index')


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 6))
plt.subplots_adjust(wspace=0.3)

# Uncorrelated Species Thermo Uncertainties
mat1 = axs[0].matshow(uncorrelated_kinetics_cov)
axs[0].xaxis.set_ticks_position('bottom')
axs[0].xaxis.set_label_position('bottom')
cb1 = fig.colorbar(mat1, ax=axs[0], fraction=0.046, pad=0.04)
cb1.ax.set_title('    (ln k)^2')
axs[0].set_title('Uncorrelated\nKinetics Covariance Matrix')
axs[0].set_xlabel('Reaction Index')
axs[0].set_ylabel('Reaction Index')

# Correlated Species Thermo Uncertainties
mat2 = axs[1].matshow(correlated_kinetics_cov)
axs[1].xaxis.set_ticks_position('bottom')
axs[1].xaxis.set_label_position('bottom')
cb2 = fig.colorbar(mat2, ax=axs[1], fraction=0.046, pad=0.04)
cb2.ax.set_title('    (ln k)^2')
axs[1].set_title('Correlated\nKinetics Covariance Matrix')
axs[1].set_xlabel('Reaction Index')
axs[1].set_ylabel('Reaction Index')
